In [1]:
%matplotlib inline
import cv2
import sys

import datacube
import numpy as np
import pandas as pd

sys.path.insert(1, '/home/jovyan/dev/Tools')
from dea_tools.plotting import display_map
from dea_tools.landcover import get_colour_scheme, _get_layer_name, lc_colourmap
from matplotlib import colors as mcolours
import rioxarray
import odc
import time

import matplotlib.pyplot as plt

In [2]:
dc = datacube.Datacube(app="DEA_Land_Cover")

In [3]:
product = "ga_ls_landcover_class_cyear_3"

measurements = dc.list_measurements()
measurements.loc[product]

,name,dtype,units,nodata,aliases,flags_definition
measurement,,,,,,
level3,level3,uint8,1,255,NaN,NaN
level4,level4,uint8,1,255,[full_classification],NaN


In [4]:
for year in reversed(range(1988,1989)):
    print(year)
    
    # Create the 'query' dictionary object, which contains the longitudes, latitudes and time defined above
    query = {
        "time": str(year),
    }
    start_time = time.time()
    # Load DEA Land Cover data from the datacube
    lc = dc.load(
        product="ga_ls_landcover_class_cyear_3",
        output_crs="EPSG:3577",
        measurements=[
            # "level3",
            "level4"
        ],
        resolution=(-30, 30),
        #resampling='mode',
        **query
    )

    end_time = time.time()
    print(end_time - start_time)
        

1988
426.89011430740356


In [5]:
lc

<xarray.Dataset> Size: 18GB
Dimensions:      (time: 1, y: 128000, x: 137600)
Coordinates:
  * time         (time) datetime64[ns] 8B 1988-07-01T23:59:59.999999
  * y            (y) float64 1MB -1.056e+06 -1.056e+06 ... -4.896e+06 -4.896e+06
  * x            (x) float64 1MB -1.92e+06 -1.92e+06 ... 2.208e+06 2.208e+06
    spatial_ref  int32 4B 3577
Data variables:
    level4       (time, y, x) uint8 18GB 255 255 255 255 255 ... 255 255 255 255
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [ ]:
start_time = time.time()

odc.geo.xr.write_cog(
    lc.level4,
    "test_COG_30m_deflate_z9_6overview_bilinear.tif",
    overwrite=True,
    dtype='uint16',
    compress='deflate', #'deflate','zstd'  # Compression type
    zlevel=9,  # compression level
    blocksize=1024,  # block size (chunks)
    overview_levels=[2, 4, 8, 16, 32, 64], # number of overviews (different levels of zoom-out)
    overview_resampling='bilinear',  # resampling method for overviews 'bilinear', 'cubic','nearest', 'mode'
    tiled=True,  # enable tiling, for easier/quicker reading of COG
    lock=True  # False: allow multiple threads to write to the file simultaneously, True: no multiple threads, slower but more robust
)


end_time = time.time()
end_time - start_time

In [ ]:
start_time = time.time()

odc.geo.xr.write_cog(
    lc.level4,
    "test_COG_30m_deflate_z9_6overview_bilinear.tif",
    overwrite=True,
    dtype='uint16',
    compress='deflate', #'deflate','zstd'  # Compression type
    zlevel=9,  # compression level
    blocksize=1024,  # block size (chunks)
    overview_levels=[2, 4, 8, 16, 32, 64], # number of overviews (different levels of zoom-out)
    overview_resampling='nearest',  # resampling method for overviews 'bilinear', 'cubic','nearest', 'mode'
    tiled=True,  # enable tiling, for easier/quicker reading of COG
    lock=True  # False: allow multiple threads to write to the file simultaneously, True: no multiple threads, slower but more robust
)


end_time = time.time()
end_time - start_time